# SentiVox — Multilingual Sentiment Analysis Model Training
Fine-tunes google/muril-base-cased on GoEmotions for multi-label emotion
classification and document-level sentiment analysis.

Output format:
```
{
    "sentiment": "positive" | "negative" | "neutral" | "ambiguous",
    "emotions": ["<GoEmotions label>", ...],
    "confidence": { "<emotion>": <float>, ... }
}
```

Designed for Google Colab (free-tier GPU).
================================================================================


# 0. Install Dependencies (run once on Colab)


In [ ]:
DATASET_NAME = "goemotions-eng.json"
# If downloading dataset from github then make sure to access that dataset from your 
# browswer first and then copy the full URL containing parameter ?token=... in the URL
URL_OF_DATASET = "https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/dataset/cleaned/goemotions-eng.json"

!wget -O $DATASET_NAME $URL_OF_DATASET

%pip install transformers datasets accelerate scikit-learn torch numpy pandas matplotlib seaborn

!ls


# 1. Imports


In [ ]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    multilabel_confusion_matrix,
)
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend (safe for Colab/headless)
import matplotlib.pyplot as plt
import seaborn as sns



# 2. Configuration


In [ ]:
class Config:
    """Central configuration for the training pipeline."""
    # Model
    MODEL_NAME = "google/muril-base-cased"
    MAX_LENGTH = 128               # Token limit per input
    NUM_LABELS = 28                # 27 GoEmotions + neutral

    # Training hyperparameters (tuned for Colab free-tier T4 GPU)
    BATCH_SIZE = 32
    GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size = 64
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    NUM_EPOCHS = 7
    WARMUP_RATIO = 0.1
    FP16 = True                    # Mixed precision for faster training

    # Early stopping
    EARLY_STOPPING_PATIENCE = 2
    METRIC_FOR_BEST_MODEL = "f1_macro"

    # Paths
    OUTPUT_DIR = "./models/sentivox-muril-goemotions"
    LOGGING_DIR = "./models/logs"
    RESULTS_DIR = "./models/results"

    # Sentiment classification threshold
    EMOTION_THRESHOLD = 0.5        # Sigmoid threshold for multi-label

config = Config()

# Create output directories
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.LOGGING_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)


# 3. Load & Split Data (Existing Code — Preserved)


In [ ]:
TRAINING_DATA = []
VALIDATION_DATA = []
TESTING_DATA = []

with open(DATASET_NAME, "r", encoding="utf-8") as file:
    print("Successfully loaded Dataset File")
    from json import load
    lst = load(file)
    print("Successfully Parsed Dataset")

    training_data_size = int(len(lst) * 0.6)
    validation_data_size = int(len(lst) * 0.8)
    """
        FORMAT OF DATA:
        {
            "text": "",
            "emotions": []
        }
    """
    TRAINING_DATA.extend(lst[:training_data_size])
    print("Succesfully created sample of 'TRAINING DATA'")
    VALIDATION_DATA.extend(lst[training_data_size:validation_data_size])
    print("Succesfully created sample of 'VALIDATION DATA'")
    TESTING_DATA.extend(lst[validation_data_size:])
    print("Succesfully created sample of 'TESTING DATA'")

print("\nSizes of Samples:")
print("      Training Data: ", len(TRAINING_DATA))
print("      Validation Data: ", len(VALIDATION_DATA))
print("      Testing Data: ", len(TESTING_DATA))

print("\n Glimpse of Data:")
print("      Training Data: ", TRAINING_DATA[:2])
print("      Validation Data: ", VALIDATION_DATA[:2])
print("      Testing Data: ", TESTING_DATA[:2])


# LABELS IN DATASET

In [ ]:
EMOTION_LABELS = ['admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral']


# 4. Emotion → Sentiment Mapping

- ### Maps each of the 28 GoEmotions labels into sentiment buckets.
- ### Reference: GoEmotions taxonomy (Demszky et al., 2020)

In [ ]:
EMOTION_TO_SENTIMENT = {
    # Positive emotions
    "admiration":  "positive",
    "amusement":   "positive",
    "approval":    "positive",
    "caring":      "positive",
    "desire":      "positive",
    "excitement":  "positive",
    "gratitude":   "positive",
    "joy":         "positive",
    "love":        "positive",
    "optimism":    "positive",
    "pride":       "positive",
    "relief":      "positive",
    # Negative emotions
    "anger":          "negative",
    "annoyance":      "negative",
    "disappointment": "negative",
    "disapproval":    "negative",
    "disgust":        "negative",
    "embarrassment":  "negative",
    "fear":           "negative",
    "grief":          "negative",
    "nervousness":    "negative",
    "remorse":        "negative",
    "sadness":        "negative",
    # Ambiguous emotions (can be positive or negative depending on context)
    "confusion":   "ambiguous",
    "curiosity":   "ambiguous",
    "realization": "ambiguous",
    "surprise":    "ambiguous",
    # Neutral
    "neutral":     "neutral",
}

SENTIMENT_LABELS = ["positive", "negative", "neutral", "ambiguous"]

def emotions_to_sentiment(emotion_list: list[str]) -> str:
    """
    Derive document-level sentiment from a list of predicted emotions.

    Logic:
      1. Collect the sentiment bucket for each predicted emotion.
      2. If only one unique bucket → return it.
      3. If both positive and negative are present → "ambiguous".
      4. Otherwise, return the majority bucket.
    """
    if not emotion_list:
        return "neutral"

    buckets = [EMOTION_TO_SENTIMENT.get(e, "neutral") for e in emotion_list]
    unique = set(buckets)

    if len(unique) == 1:
        return unique.pop()
    if "positive" in unique and "negative" in unique:
        return "ambiguous"

    # Majority vote (excluding ambiguous for tie-breaking)
    from collections import Counter
    counts = Counter(buckets)
    # Remove ambiguous from majority voting — it shouldn't dominate
    if len(counts) > 1:
        counts.pop("ambiguous", None)
    return counts.most_common(1)[0][0]


# 5. PyTorch Dataset


In [ ]:
print("\n[Step 5] Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

class GoEmotionsDataset(Dataset):
    """
    Multi-label dataset for GoEmotions.

    Each sample is tokenized and converted to a binary label vector of
    length 28 (one-hot over EMOTION_LABELS).
    """

    def __init__(self, data: list[dict], tokenizer, max_length: int):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label2id = {label: idx for idx, label in enumerate(EMOTION_LABELS)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        emotions = item["emotions"]

        # Tokenize
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Multi-label binary vector
        labels = torch.zeros(len(EMOTION_LABELS), dtype=torch.float)
        for emotion in emotions:
            if emotion in self.label2id:
                labels[self.label2id[emotion]] = 1.0

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }


print("[Step 5] Building datasets...")
train_dataset = GoEmotionsDataset(TRAINING_DATA, tokenizer, config.MAX_LENGTH)
val_dataset = GoEmotionsDataset(VALIDATION_DATA, tokenizer, config.MAX_LENGTH)
test_dataset = GoEmotionsDataset(TESTING_DATA, tokenizer, config.MAX_LENGTH)
print(f"  Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


# 6. Model Initialization


In [ ]:
print("\n[Step 6] Loading MuRIL model for multi-label classification...")
model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=config.NUM_LABELS,
    problem_type="multi_label_classification",
    id2label={i: label for i, label in enumerate(EMOTION_LABELS)},
    label2id={label: i for i, label in enumerate(EMOTION_LABELS)},
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Using device: {device}")
model.to(device)

# Print model parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")


# 7. Metrics Computation


In [ ]:
def compute_metrics(eval_pred):
    """
    Compute multi-label classification metrics for the Trainer.

    Uses sigmoid threshold to convert logits → binary predictions.
    Returns accuracy, F1 (macro & micro), precision, and recall.
    """
    logits, labels = eval_pred
    # Sigmoid + threshold
    probs = 1 / (1 + np.exp(-logits))  # Sigmoid
    preds = (probs >= config.EMOTION_THRESHOLD).astype(int)
    labels = labels.astype(int)

    # Sample-level exact match accuracy
    accuracy = accuracy_score(labels, preds)

    # Macro & micro F1
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)

    # Precision & Recall
    precision_macro = precision_score(labels, preds, average="macro", zero_division=0)
    recall_macro = recall_score(labels, preds, average="macro", zero_division=0)
    precision_micro = precision_score(labels, preds, average="micro", zero_division=0)
    recall_micro = recall_score(labels, preds, average="micro", zero_division=0)

    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
    }


# 8. Training Arguments & Trainer


In [ ]:
print("\n[Step 8] Configuring training...")

training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    #overwrite_output_dir=True,

    # Training
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE * 2,  # Larger batch for eval
    gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    # warmup_ratio=config.WARMUP_RATIO,
    fp16=config.FP16 and torch.cuda.is_available(),

    # Evaluation & Saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=config.METRIC_FOR_BEST_MODEL,
    greater_is_better=True,
    save_total_limit=2,              # Keep only last 2 checkpoints (saves disk)

    # Logging
    #logging_dir=config.LOGGING_DIR,
    logging_steps=100,
    report_to="none",                # Disable W&B/MLflow on Colab

    # Performance
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # Reproducibility
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=config.EARLY_STOPPING_PATIENCE)
    ],
)


# 9. Train


In [ ]:
print("\n" + "=" * 60)
print("  STARTING TRAINING")
print("=" * 60)

train_result = trainer.train()

# Log training results
print("\n[Training Complete]")
print(f"  Training loss:       {train_result.training_loss:.4f}")
print(f"  Training runtime:    {train_result.metrics['train_runtime']:.1f}s")
print(f"  Samples/second:      {train_result.metrics['train_samples_per_second']:.1f}")


# 10. Evaluate on Validation Set


In [ ]:
print("\n[Step 10] Evaluating on validation set...")
val_metrics = trainer.evaluate()
print("\nValidation Metrics:")
for key, value in val_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")


# 11. Evaluate on Test Set — Full Report


In [ ]:
print("\n[Step 11] Evaluating on test set...")
test_output = trainer.predict(test_dataset)
test_metrics = test_output.metrics

print("\nTest Metrics:")
for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

# --- Per-label classification report ---
logits = test_output.predictions
probs = 1 / (1 + np.exp(-logits))
preds = (probs >= config.EMOTION_THRESHOLD).astype(int)
true_labels = test_output.label_ids.astype(int)

print("\n" + "=" * 60)
print("  PER-LABEL CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(
    true_labels,
    preds,
    target_names=EMOTION_LABELS,
    zero_division=0,
)
print(report)

# Save the classification report to file
with open(os.path.join(config.RESULTS_DIR, "classification_report.txt"), "w") as f:
    f.write("SentiVox — Test Set Classification Report\n")
    f.write("=" * 60 + "\n")
    f.write(report)


# 12. Confusion Matrix Visualization


In [ ]:
print("\n[Step 12] Generating confusion matrices...")

def plot_multilabel_confusion_matrices(y_true, y_pred, labels, save_path):
    """
    Generate and save a grid of per-label confusion matrices.
    For multi-label problems, each label has its own 2×2 confusion matrix.
    """
    mcm = multilabel_confusion_matrix(y_true, y_pred)

    n_labels = len(labels)
    cols = 7
    rows = (n_labels + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    fig.suptitle("Per-Emotion Confusion Matrices", fontsize=16, fontweight="bold")

    for i, (cm, label) in enumerate(zip(mcm, labels)):
        ax = axes[i // cols][i % cols] if rows > 1 else axes[i % cols]
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Neg", "Pos"],
            yticklabels=["Neg", "Pos"],
            ax=ax, cbar=False,
        )
        ax.set_title(label, fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")

    # Hide unused subplots
    for i in range(n_labels, rows * cols):
        ax = axes[i // cols][i % cols] if rows > 1 else axes[i % cols]
        ax.set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Confusion matrices saved → {save_path}")


def plot_sentiment_confusion_matrix(y_true_emotions, y_pred_emotions, labels, save_path):
    """
    Build a sentiment-level confusion matrix by mapping predicted emotions
    to sentiment buckets.
    """
    true_sentiments = []
    pred_sentiments = []

    for true_vec, pred_vec in zip(y_true_emotions, y_pred_emotions):
        true_emos = [labels[j] for j, v in enumerate(true_vec) if v == 1]
        pred_emos = [labels[j] for j, v in enumerate(pred_vec) if v == 1]
        true_sentiments.append(emotions_to_sentiment(true_emos))
        pred_sentiments.append(emotions_to_sentiment(pred_emos))

    cm = confusion_matrix(true_sentiments, pred_sentiments, labels=SENTIMENT_LABELS)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Purples",
        xticklabels=SENTIMENT_LABELS,
        yticklabels=SENTIMENT_LABELS,
        ax=ax,
    )
    ax.set_title("Sentiment-Level Confusion Matrix", fontsize=14, fontweight="bold")
    ax.set_xlabel("Predicted Sentiment")
    ax.set_ylabel("True Sentiment")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Sentiment confusion matrix saved → {save_path}")

    # Also print sentiment-level classification report
    print("\n  Sentiment-Level Classification Report:")
    print(classification_report(
        true_sentiments, pred_sentiments,
        labels=SENTIMENT_LABELS, zero_division=0
    ))


# Generate both confusion matrices
emotion_cm_path = os.path.join(config.RESULTS_DIR, "confusion_matrix_emotions.png")
sentiment_cm_path = os.path.join(config.RESULTS_DIR, "confusion_matrix_sentiment.png")

plot_multilabel_confusion_matrices(true_labels, preds, EMOTION_LABELS, emotion_cm_path)
plot_sentiment_confusion_matrix(true_labels, preds, EMOTION_LABELS, sentiment_cm_path)


# 13. Save Metrics to JSON


In [ ]:
print("\n[Step 13] Saving metrics...")

all_metrics = {
    "language": "English",
    "base_model": config.MODEL_NAME,
    "dataset": "GoEmotions",
    "training": {
        "epochs": config.NUM_EPOCHS,
        "batch_size": config.BATCH_SIZE,
        "effective_batch_size": config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": config.LEARNING_RATE,
        "training_loss": train_result.training_loss,
    },
    "validation_metrics": {
        k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)
    },
    "test_metrics": {
        k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)
    },
}

metrics_path = os.path.join(config.RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"  Metrics saved → {metrics_path}")


# 14. Save Fine-Tuned Model (HuggingFace-deployable)


In [ ]:
print("\n[Step 14] Saving fine-tuned model...")

# Save model + tokenizer (HuggingFace format — ready for Hub push)
trainer.save_model(config.OUTPUT_DIR)
tokenizer.save_pretrained(config.OUTPUT_DIR)

# Save model card for HuggingFace Hub
model_card = """---
language:
  - en
  - hi
  - bn
  - ta
  - te
  - ml
  - kn
  - gu
  - mr
  - pa
  - or
  - as
  - ur
license: apache-2.0
tags:
  - sentiment-analysis
  - multilingual
  - muril
  - goemotions
  - multi-label
  - indian-languages
datasets:
  - go_emotions
pipeline_tag: text-classification
---

# SentiVox — Multilingual Sentiment Analysis Model

Fine-tuned [google/muril-base-cased](https://huggingface.co/google/muril-base-cased)
on the [GoEmotions](https://github.com/google-research/google-research/tree/master/goemotions)
dataset for multi-label emotion classification and document-level sentiment analysis.

## Model Description

- **Base Model**: google/muril-base-cased (BERT-based, supports 17 Indian languages + English)
- **Task**: Multi-label emotion classification → sentiment derivation
- **Labels**: 28 emotion labels (27 GoEmotions + neutral)
- **Sentiment Output**: positive, negative, neutral, ambiguous

## Usage

```python
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="<your-hub-id>/sentivox-muril-goemotions",
    top_k=None,
)
result = classifier("I am so happy and grateful!")
```

## Training Details

- **Dataset**: GoEmotions (English)
- **Epochs**: {epochs}
- **Batch Size**: {batch_size} (effective: {eff_batch})
- **Learning Rate**: {lr}
- **Optimizer**: AdamW with linear warmup
- **Hardware**: Google Colab (T4 GPU)

## Future Work

- Fine-tune on Hindi, Bengali, Tamil, and other Indian language datasets
- Add document-level context aggregation for longer texts
""".format(
    epochs=config.NUM_EPOCHS,
    batch_size=config.BATCH_SIZE,
    eff_batch=config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS,
    lr=config.LEARNING_RATE,
)

with open(os.path.join(config.OUTPUT_DIR, "README.md"), "w") as f:
    f.write(model_card)

print(f"  Model saved → {config.OUTPUT_DIR}")
print(f"  Model card saved → {os.path.join(config.OUTPUT_DIR, 'README.md')}")


# 15. Inference Utilities


In [ ]:
print("\n[Step 15] Setting up inference utilities...")


def load_sentivox_model(model_path: str = config.OUTPUT_DIR):
    """
    Load the fine-tuned SentiVox model and tokenizer.

    Args:
        model_path: Path to the saved model directory.

    Returns:
        Tuple of (model, tokenizer, device).
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
    loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)
    loaded_model.to(device)
    loaded_model.eval()
    return loaded_model, loaded_tokenizer, device


def predict(
    text: str,
    model_instance=None,
    tokenizer_instance=None,
    device_instance=None,
    threshold: float = config.EMOTION_THRESHOLD,
) -> dict:
    """
    Run inference on a single text sample.

    Args:
        text:               Input text string.
        model_instance:     Loaded PyTorch model (uses global if None).
        tokenizer_instance: Loaded tokenizer (uses global if None).
        device_instance:    Torch device (uses global if None).
        threshold:          Sigmoid threshold for emotion activation.

    Returns:
        Dictionary with keys:
            - "sentiment": str — positive/negative/neutral/ambiguous
            - "emotions": list[str] — activated emotion labels
            - "confidence": dict[str, float] — per-emotion probabilities
    """
    _model = model_instance or model
    _tokenizer = tokenizer_instance or tokenizer
    _device = device_instance or device

    encoding = _tokenizer(
        text,
        max_length=config.MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    encoding = {k: v.to(_device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = _model(**encoding)
        logits = outputs.logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    # Threshold to get active emotions
    active_emotions = []
    confidence = {}
    for idx, (label, prob) in enumerate(zip(EMOTION_LABELS, probs)):
        confidence[label] = round(float(prob), 4)
        if prob >= threshold:
            active_emotions.append(label)

    # If no emotion passes threshold, take the highest one
    if not active_emotions:
        top_idx = int(np.argmax(probs))
        active_emotions.append(EMOTION_LABELS[top_idx])

    # Derive sentiment
    sentiment = emotions_to_sentiment(active_emotions)

    return {
        "sentiment": sentiment,
        "emotions": active_emotions,
        "confidence": confidence,
    }


def predict_batch(
    texts: list[str],
    model_instance=None,
    tokenizer_instance=None,
    device_instance=None,
    threshold: float = config.EMOTION_THRESHOLD,
    batch_size: int = 32,
) -> list[dict]:
    """
    Run inference on a batch of texts.

    Args:
        texts:              List of input text strings.
        model_instance:     Loaded PyTorch model.
        tokenizer_instance: Loaded tokenizer.
        device_instance:    Torch device.
        threshold:          Sigmoid threshold.
        batch_size:         Inference batch size.

    Returns:
        List of prediction dictionaries (same format as predict()).
    """
    results = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        for text in batch_texts:
            results.append(predict(
                text, model_instance, tokenizer_instance,
                device_instance, threshold,
            ))
    return results


def predict_document(
    paragraphs: list[str],
    model_instance=None,
    tokenizer_instance=None,
    device_instance=None,
    threshold: float = config.EMOTION_THRESHOLD,
) -> dict:
    """
    Document-level sentiment analysis.

    Splits a document into paragraphs/sentences, predicts emotion for each,
    then aggregates to determine overall document sentiment.

    Args:
        paragraphs: List of text segments (paragraphs or sentences).

    Returns:
        Dictionary with:
            - "overall_sentiment": str
            - "overall_emotions": list[str] — union of all segment emotions
            - "segment_predictions": list[dict] — per-segment predictions
    """
    segment_preds = predict_batch(
        paragraphs, model_instance, tokenizer_instance,
        device_instance, threshold,
    )

    # Aggregate emotions across all segments
    all_emotions = set()
    for pred in segment_preds:
        all_emotions.update(pred["emotions"])

    overall_sentiment = emotions_to_sentiment(list(all_emotions))

    return {
        "overall_sentiment": overall_sentiment,
        "overall_emotions": sorted(list(all_emotions)),
        "segment_predictions": segment_preds,
    }


# 16. Demo Predictions


In [ ]:
print("\n" + "=" * 60)
print("  DEMO PREDICTIONS")
print("=" * 60)

demo_texts = [
    "I am so happy and grateful for everything you have done!",
    "This is absolutely terrible and disgusting behavior.",
    "I'm not sure what to think about this situation.",
    "The weather is cloudy today.",
    "I love this but I'm also terrified of what comes next.",
    "He is not bad",
    "He is not good"
]

for text in demo_texts:
    result = predict(text)
    print(f"\n  Text: \"{text}\"")
    print(f"  Sentiment: {result['sentiment']}")
    print(f"  Emotions:  {result['emotions']}")
    top_3 = sorted(result["confidence"].items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"  Top 3:     {[(e, f'{s:.3f}') for e, s in top_3]}")

# Document-level demo
print("\n" + "-" * 40)
print("  Document-Level Analysis Demo")
print("-" * 40)

doc_paragraphs = [
    "The product arrived on time and was well packaged.",
    "However, the quality was much lower than expected.",
    "I contacted support and they were incredibly helpful and resolved it quickly.",
]

doc_result = predict_document(doc_paragraphs)
print(f"\n  Overall Sentiment: {doc_result['overall_sentiment']}")
print(f"  Overall Emotions:  {doc_result['overall_emotions']}")
for i, seg in enumerate(doc_result["segment_predictions"]):
    print(f"  Segment {i + 1}: {seg['sentiment']} — {seg['emotions']}")


# 17. Summary


In [ ]:
print("\n" + "=" * 60)
print("  TRAINING PIPELINE COMPLETE")
print("=" * 60)
print(f"""
  Model:          {config.MODEL_NAME}
  Saved to:       {config.OUTPUT_DIR}
  Metrics:        {metrics_path}
  Report:         {os.path.join(config.RESULTS_DIR, 'classification_report.txt')}

  To push to HuggingFace Hub:
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_folder(
        folder_path="{config.OUTPUT_DIR}",
        repo_id="<your-username>/sentivox-muril-goemotions",
        repo_type="model",
    )

  To load for inference:
    model, tokenizer, device = load_sentivox_model("{config.OUTPUT_DIR}")
    result = predict("Your text here", model, tokenizer, device)
""")